# EMP on Transformer Attention Scores (Pre-Softmax)

This notebook implements Effective Model Pruning (EMP) on transformer attention scores **before softmax**.

We test two pruning strategies:
1. **Per-Head EMP**: Apply EMP independently to each attention head
2. **Global EMP**: Apply EMP across all attention scores (all heads combined)

## Outline
1. Setup and dependencies
2. Build a GPT-style transformer for language modeling
3. Train on WikiText-2 and evaluate perplexity
4. Implement EMP-based attention score pruning
5. Evaluate with different β values

## 1. Setup

In [1]:
import os
import math
import time
import copy
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# For WikiText-2 dataset
try:
    from datasets import load_dataset
    HAS_DATASETS = True
except ImportError:
    HAS_DATASETS = False
    print("Installing datasets library...")
    !pip install datasets -q
    from datasets import load_dataset
    HAS_DATASETS = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 5090


## 2. Model Configuration

In [2]:
@dataclass
class TransformerConfig:
    """Configuration for GPT-style transformer."""
    vocab_size: int = 50257  # GPT-2 vocab size, will be overridden
    block_size: int = 256    # context length
    n_layer: int = 6         # number of transformer blocks
    n_head: int = 8          # number of attention heads
    n_embd: int = 512        # embedding dimension
    dropout: float = 0.1
    bias: bool = True        # use bias in Linear and LayerNorm
    
    # Training config
    batch_size: int = 32
    epochs: int = 10
    lr: float = 3e-4
    weight_decay: float = 0.01
    warmup_epochs: int = 1

config = TransformerConfig()
print(f"Config: {config}")

Config: TransformerConfig(vocab_size=50257, block_size=256, n_layer=6, n_head=8, n_embd=512, dropout=0.1, bias=True, batch_size=32, epochs=10, lr=0.0003, weight_decay=0.01, warmup_epochs=1)


## 3. Dataset: WikiText-2

In [3]:
class CharTokenizer:
    """Simple character-level tokenizer."""
    def __init__(self, text: str):
        chars = sorted(list(set(text)))
        self.char_to_idx = {ch: i for i, ch in enumerate(chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(chars)}
        self.vocab_size = len(chars)
    
    def encode(self, text: str) -> List[int]:
        return [self.char_to_idx.get(ch, 0) for ch in text]
    
    def decode(self, indices: List[int]) -> str:
        return ''.join([self.idx_to_char.get(i, '') for i in indices])


class WordTokenizer:
    """Simple word-level tokenizer with special tokens."""
    def __init__(self, text: str, max_vocab: int = 30000):
        # Count word frequencies
        words = text.split()
        word_freq = {}
        for w in words:
            word_freq[w] = word_freq.get(w, 0) + 1
        
        # Keep most frequent words
        sorted_words = sorted(word_freq.items(), key=lambda x: -x[1])
        vocab_words = [w for w, _ in sorted_words[:max_vocab - 2]]
        
        # Special tokens
        self.pad_token = '<PAD>'
        self.unk_token = '<UNK>'
        
        all_tokens = [self.pad_token, self.unk_token] + vocab_words
        self.word_to_idx = {w: i for i, w in enumerate(all_tokens)}
        self.idx_to_word = {i: w for i, w in enumerate(all_tokens)}
        self.vocab_size = len(all_tokens)
        self.unk_idx = self.word_to_idx[self.unk_token]
    
    def encode(self, text: str) -> List[int]:
        return [self.word_to_idx.get(w, self.unk_idx) for w in text.split()]
    
    def decode(self, indices: List[int]) -> str:
        return ' '.join([self.idx_to_word.get(i, self.unk_token) for i in indices])

In [4]:
class LMDataset(Dataset):
    """Language Modeling Dataset."""
    def __init__(self, data: torch.Tensor, block_size: int):
        self.data = data
        self.block_size = block_size
    
    def __len__(self):
        return max(0, len(self.data) - self.block_size - 1)
    
    def __getitem__(self, idx):
        x = self.data[idx:idx + self.block_size]
        y = self.data[idx + 1:idx + self.block_size + 1]
        return x, y


def load_wikitext2(config: TransformerConfig, use_char_level: bool = False):
    """Load WikiText-2 dataset."""
    print("Loading WikiText-2...")
    dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
    
    # Combine all text
    train_text = '\n'.join(dataset['train']['text'])
    val_text = '\n'.join(dataset['validation']['text'])
    test_text = '\n'.join(dataset['test']['text'])
    
    # Create tokenizer
    if use_char_level:
        tokenizer = CharTokenizer(train_text)
    else:
        tokenizer = WordTokenizer(train_text, max_vocab=20000)
    
    print(f"Vocabulary size: {tokenizer.vocab_size}")
    
    # Encode data
    train_ids = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)
    val_ids = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)
    test_ids = torch.tensor(tokenizer.encode(test_text), dtype=torch.long)
    
    print(f"Train tokens: {len(train_ids):,}")
    print(f"Val tokens: {len(val_ids):,}")
    print(f"Test tokens: {len(test_ids):,}")
    
    # Create datasets
    train_dataset = LMDataset(train_ids, config.block_size)
    val_dataset = LMDataset(val_ids, config.block_size)
    test_dataset = LMDataset(test_ids, config.block_size)
    
    # Windows notebooks cannot fork worker processes reliably, so fall back to the main process
    worker_count = 0 if os.name == "nt" else 2
    pin_memory = torch.cuda.is_available() and worker_count > 0
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=worker_count, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=worker_count, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=worker_count, pin_memory=pin_memory)
    
    return train_loader, val_loader, test_loader, tokenizer

In [5]:
# Load data
train_loader, val_loader, test_loader, tokenizer = load_wikitext2(config, use_char_level=False)
config.vocab_size = tokenizer.vocab_size
print(f"\nUpdated vocab_size: {config.vocab_size}")

Loading WikiText-2...
Vocabulary size: 20000
Train tokens: 2,051,910
Val tokens: 213,886
Test tokens: 241,211

Updated vocab_size: 20000


## 4. Transformer Model with EMP-compatible Attention

In [6]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with EMP support.
    
    Supports pruning attention scores (pre-softmax) via EMP.
    """
    
    def __init__(self, config: TransformerConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.dropout = config.dropout
        
        # QKV projection
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        
        # Regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        
        # Causal mask
        self.register_buffer("mask", torch.tril(torch.ones(config.block_size, config.block_size))
                                          .view(1, 1, config.block_size, config.block_size))
        
        # EMP pruning config (set externally)
        self.emp_beta = None  # If set, apply EMP
        self.emp_mode = 'per_head'  # 'per_head' or 'global'
        
        # Storage for attention scores (for analysis)
        self.last_attn_scores = None
        self.last_attn_mask = None
    
    def compute_neff(self, scores: torch.Tensor) -> int:
        """Compute effective number N_eff from attention scores.
        
        Args:
            scores: Attention scores of shape (*, N)
        
        Returns:
            N_eff: Effective number of significant entries
        """
        # Flatten to 1D for N_eff computation
        s = scores.reshape(-1)
        
        # Normalize by absolute sum
        s_abs = torch.abs(s)
        s_sum = s_abs.sum()
        if s_sum < 1e-10:
            return len(s)
        
        omega = s_abs / s_sum
        
        # N_eff = 1 / sum(omega^2)
        neff = 1.0 / (omega ** 2).sum()
        return neff
    
    def apply_emp_mask(self, scores: torch.Tensor, beta: float, mode: str = 'per_head') -> Tuple[torch.Tensor, torch.Tensor]:
        """Apply EMP-based masking to attention scores.
        
        Args:
            scores: Pre-softmax attention scores (B, n_head, T, T)
            beta: EMP coefficient
            mode: 'per_head' or 'global'
        
        Returns:
            masked_scores: Scores with low-importance entries set to -inf
            mask: Binary mask indicating retained entries
        """
        B, H, T, T2 = scores.shape
        
        if mode == 'per_head':
            # Apply EMP independently per head
            masks = []
            for h in range(H):
                head_scores = scores[:, h, :, :]  # (B, T, T)
                
                # Compute N_eff for this head across batch
                neff = self.compute_neff(head_scores)
                r_neff = int(torch.floor(beta * neff).clamp(1, head_scores.numel()))
                
                # Find threshold
                flat_scores = head_scores.reshape(-1)
                if r_neff >= len(flat_scores):
                    head_mask = torch.ones_like(head_scores, dtype=torch.bool)
                else:
                    _, indices = torch.sort(torch.abs(flat_scores), descending=True)
                    thresh = torch.abs(flat_scores)[indices[r_neff - 1]]
                    head_mask = torch.abs(head_scores) >= thresh
                
                masks.append(head_mask.unsqueeze(1))
            
            mask = torch.cat(masks, dim=1)  # (B, H, T, T)
        
        elif mode == 'global':
            # Apply EMP globally across all heads
            neff = self.compute_neff(scores)
            r_neff = int(torch.floor(beta * neff).clamp(1, scores.numel()))
            
            flat_scores = scores.reshape(-1)
            if r_neff >= len(flat_scores):
                mask = torch.ones_like(scores, dtype=torch.bool)
            else:
                _, indices = torch.sort(torch.abs(flat_scores), descending=True)
                thresh = torch.abs(flat_scores)[indices[r_neff - 1]]
                mask = torch.abs(scores) >= thresh
        
        else:
            raise ValueError(f"Unknown EMP mode: {mode}")
        
        # Apply mask: set pruned entries to -inf so they become 0 after softmax
        masked_scores = scores.clone()
        masked_scores[~mask] = float('-inf')
        
        return masked_scores, mask
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        
        # QKV projection
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        
        # Reshape for multi-head attention
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))  # (B, H, T, T)
        
        # Apply causal mask
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        
        # Store pre-softmax scores for analysis
        self.last_attn_scores = scores.detach().clone()
        
        # Apply EMP if configured
        if self.emp_beta is not None and self.emp_beta > 0:
            scores, emp_mask = self.apply_emp_mask(scores, self.emp_beta, self.emp_mode)
            self.last_attn_mask = emp_mask
        else:
            self.last_attn_mask = None
        
        # Softmax and dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        
        # Apply attention
        out = attn_weights @ v  # (B, H, T, D)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        
        # Output projection
        out = self.resid_dropout(self.c_proj(out))
        
        return out

In [7]:
class MLP(nn.Module):
    """Feed-forward network."""
    
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class TransformerBlock(nn.Module):
    """Transformer block with pre-LayerNorm."""
    
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [8]:
class GPT(nn.Module):
    """GPT-style Language Model."""
    
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.config = config
        
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # Weight tying
        self.transformer.wte.weight = self.lm_head.weight
        
        # Initialize weights
        self.apply(self._init_weights)
        
        # Count parameters
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model parameters: {n_params/1e6:.2f}M")
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None):
        B, T = idx.shape
        assert T <= self.config.block_size, f"Sequence length {T} > block_size {self.config.block_size}"
        
        # Embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.transformer.wte(idx)  # (B, T, C)
        pos_emb = self.transformer.wpe(pos)  # (T, C)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        # Transformer blocks
        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        
        # Output
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss
    
    def set_emp(self, beta: Optional[float], mode: str = 'per_head'):
        """Set EMP pruning for all attention layers.
        
        Args:
            beta: EMP coefficient. None or 0 to disable.
            mode: 'per_head' or 'global'
        """
        for block in self.transformer.h:
            block.attn.emp_beta = beta
            block.attn.emp_mode = mode
    
    def get_attention_stats(self) -> Dict:
        """Get attention statistics from all layers."""
        stats = {}
        for i, block in enumerate(self.transformer.h):
            attn = block.attn
            if attn.last_attn_scores is not None:
                scores = attn.last_attn_scores
                stats[f'layer_{i}'] = {
                    'scores_mean': scores.mean().item(),
                    'scores_std': scores.std().item(),
                    'scores_min': scores[scores != float('-inf')].min().item() if (scores != float('-inf')).any() else 0,
                    'scores_max': scores[scores != float('-inf')].max().item() if (scores != float('-inf')).any() else 0,
                }
                if attn.last_attn_mask is not None:
                    mask = attn.last_attn_mask
                    stats[f'layer_{i}']['sparsity'] = 1.0 - mask.float().mean().item()
        return stats

In [9]:
# Build model
model = GPT(config).to(device)
print(f"\nModel architecture:")
print(model)

Model parameters: 29.29M

Model architecture:
GPT(
  (transformer): ModuleDict(
    (wte): Embedding(20000, 512)
    (wpe): Embedding(256, 512)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x TransformerBlock(
        (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=512, out_features=1536, bias=True)
          (c_proj): Linear(in_features=512, out_features=512, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=512, out_features=2048, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((512,

## 5. Training

In [10]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Tuple[float, float]:
    """Evaluate model and return average loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()
    
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    
    return avg_loss, perplexity


def train_epoch(model: nn.Module, loader: DataLoader, optimizer, scheduler=None) -> float:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    total_tokens = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()
    
    if scheduler is not None:
        scheduler.step()
    
    return total_loss / total_tokens

In [11]:
def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, 
                config: TransformerConfig, save_path: str):
    """Full training loop."""
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95)
    )
    
    # Scheduler
    warmup_scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, 
                                 total_iters=config.warmup_epochs)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=config.epochs - config.warmup_epochs)
    scheduler = SequentialLR(optimizer, [warmup_scheduler, cosine_scheduler], 
                              milestones=[config.warmup_epochs])
    
    best_val_ppl = float('inf')
    
    print(f"\nStarting training for {config.epochs} epochs...")
    print("=" * 80)
    
    for epoch in range(config.epochs):
        start_time = time.time()
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, scheduler)
        train_ppl = math.exp(train_loss)
        
        # Evaluate
        val_loss, val_ppl = evaluate(model, val_loader)
        
        elapsed = time.time() - start_time
        lr = optimizer.param_groups[0]['lr']
        
        # Save best model
        if val_ppl < best_val_ppl:
            best_val_ppl = val_ppl
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save({
                'model': model.state_dict(),
                'config': config.__dict__,
                'epoch': epoch,
                'val_ppl': val_ppl,
            }, save_path)
            marker = " *"
        else:
            marker = ""
        
        print(f"Epoch {epoch+1:3d}/{config.epochs} | "
              f"Train Loss: {train_loss:.4f} PPL: {train_ppl:8.2f} | "
              f"Val Loss: {val_loss:.4f} PPL: {val_ppl:8.2f} | "
              f"LR: {lr:.2e} | Time: {elapsed:.1f}s{marker}")
    
    print("=" * 80)
    print(f"Best validation perplexity: {best_val_ppl:.2f}")
    
    return best_val_ppl

In [ ]:
# Training
save_path = "./checkpoints/GPT_WikiText2_best.pth"

# Check if already trained
if os.path.exists(save_path):
    print(f"Loading pre-trained model from {save_path}")
    checkpoint = torch.load(save_path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded model with val PPL: {checkpoint['val_ppl']:.2f}")
else:
    best_ppl = train_model(model, train_loader, val_loader, config, save_path)


Starting training for 10 epochs...


In [ ]:
# Evaluate baseline (no EMP)
model.set_emp(None)
val_loss, val_ppl = evaluate(model, val_loader)
test_loss, test_ppl = evaluate(model, test_loader)

print(f"\n{'='*60}")
print(f"BASELINE RESULTS (No EMP)")
print(f"{'='*60}")
print(f"Validation Loss: {val_loss:.4f} | Perplexity: {val_ppl:.2f}")
print(f"Test Loss:       {test_loss:.4f} | Perplexity: {test_ppl:.2f}")
print(f"{'='*60}")

## 6. EMP Attention Pruning Experiments

In [ ]:
def analyze_attention_emp(model: nn.Module, loader: DataLoader, beta: float, mode: str) -> Dict:
    """Analyze EMP pruning effect on attention."""
    model.eval()
    model.set_emp(beta, mode)
    
    # Get one batch to analyze
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    
    with torch.no_grad():
        _ = model(x)
    
    stats = model.get_attention_stats()
    
    # Compute average sparsity across layers
    sparsities = [s.get('sparsity', 0) for s in stats.values()]
    avg_sparsity = sum(sparsities) / len(sparsities) if sparsities else 0
    
    return {
        'layer_stats': stats,
        'avg_sparsity': avg_sparsity
    }

In [ ]:
def run_emp_experiments(model: nn.Module, val_loader: DataLoader, test_loader: DataLoader,
                        beta_list: List[float], modes: List[str]):
    """Run EMP experiments with different beta values and modes."""
    
    results = []
    
    # Baseline
    model.set_emp(None)
    baseline_val_loss, baseline_val_ppl = evaluate(model, val_loader)
    baseline_test_loss, baseline_test_ppl = evaluate(model, test_loader)
    
    results.append({
        'mode': 'baseline',
        'beta': 0,
        'val_loss': baseline_val_loss,
        'val_ppl': baseline_val_ppl,
        'test_loss': baseline_test_loss,
        'test_ppl': baseline_test_ppl,
        'sparsity': 0.0,
        'delta_val_ppl': 0.0,
        'delta_test_ppl': 0.0,
    })
    
    print(f"\n{'='*100}")
    print(f"EMP ATTENTION PRUNING EXPERIMENTS")
    print(f"{'='*100}")
    print(f"Baseline - Val PPL: {baseline_val_ppl:.2f} | Test PPL: {baseline_test_ppl:.2f}")
    print(f"{'='*100}")
    
    for mode in modes:
        print(f"\n--- Mode: {mode.upper()} ---")
        print(f"{'Beta':>8} | {'Val Loss':>10} | {'Val PPL':>10} | {'Test Loss':>10} | {'Test PPL':>10} | "
              f"{'Sparsity':>10} | {'ΔPPL (Val)':>12} | {'ΔPPL (Test)':>12}")
        print("-" * 110)
        
        for beta in beta_list:
            # Set EMP
            model.set_emp(beta, mode)
            
            # Evaluate
            val_loss, val_ppl = evaluate(model, val_loader)
            test_loss, test_ppl = evaluate(model, test_loader)
            
            # Analyze sparsity
            analysis = analyze_attention_emp(model, val_loader, beta, mode)
            sparsity = analysis['avg_sparsity']
            
            delta_val = val_ppl - baseline_val_ppl
            delta_test = test_ppl - baseline_test_ppl
            
            results.append({
                'mode': mode,
                'beta': beta,
                'val_loss': val_loss,
                'val_ppl': val_ppl,
                'test_loss': test_loss,
                'test_ppl': test_ppl,
                'sparsity': sparsity,
                'delta_val_ppl': delta_val,
                'delta_test_ppl': delta_test,
            })
            
            print(f"{beta:8.2f} | {val_loss:10.4f} | {val_ppl:10.2f} | {test_loss:10.4f} | {test_ppl:10.2f} | "
                  f"{sparsity*100:9.2f}% | {delta_val:+12.2f} | {delta_test:+12.2f}")
    
    # Reset to baseline
    model.set_emp(None)
    
    return results

In [ ]:
# Run experiments
beta_list = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
modes = ['per_head', 'global']

results = run_emp_experiments(model, val_loader, test_loader, beta_list, modes)

In [ ]:
# Create results DataFrame
import pandas as pd

df = pd.DataFrame(results)
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Colors for modes
colors = {'per_head': 'blue', 'global': 'red'}
markers = {'per_head': 'o', 'global': 's'}

for mode in modes:
    mode_df = df[df['mode'] == mode]
    
    # Plot 1: PPL vs Beta
    axes[0].plot(mode_df['beta'], mode_df['test_ppl'], 
                 color=colors[mode], marker=markers[mode], label=mode)

# Add baseline
baseline_ppl = df[df['mode'] == 'baseline']['test_ppl'].values[0]
axes[0].axhline(y=baseline_ppl, color='green', linestyle='--', label='baseline')
axes[0].set_xlabel('Beta')
axes[0].set_ylabel('Test Perplexity')
axes[0].set_title('Test Perplexity vs Beta')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for mode in modes:
    mode_df = df[df['mode'] == mode]
    
    # Plot 2: Sparsity vs Beta
    axes[1].plot(mode_df['beta'], mode_df['sparsity'] * 100, 
                 color=colors[mode], marker=markers[mode], label=mode)

axes[1].set_xlabel('Beta')
axes[1].set_ylabel('Attention Sparsity (%)')
axes[1].set_title('Attention Sparsity vs Beta')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for mode in modes:
    mode_df = df[df['mode'] == mode]
    
    # Plot 3: Delta PPL vs Sparsity
    axes[2].plot(mode_df['sparsity'] * 100, mode_df['delta_test_ppl'], 
                 color=colors[mode], marker=markers[mode], label=mode)

axes[2].axhline(y=0, color='green', linestyle='--', label='baseline')
axes[2].set_xlabel('Attention Sparsity (%)')
axes[2].set_ylabel('ΔPPL (Test)')
axes[2].set_title('PPL Change vs Sparsity')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./checkpoints/emp_attention_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved to ./checkpoints/emp_attention_results.png")

## 7. Per-Layer Analysis

In [ ]:
def analyze_per_layer_emp(model: nn.Module, loader: DataLoader, beta: float = 1.0):
    """Detailed per-layer analysis of EMP attention pruning."""
    model.eval()
    model.set_emp(beta, 'per_head')
    
    # Get one batch
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    
    with torch.no_grad():
        _ = model(x)
    
    print(f"\n{'='*80}")
    print(f"PER-LAYER ATTENTION ANALYSIS (beta={beta})")
    print(f"{'='*80}")
    
    for i, block in enumerate(model.transformer.h):
        attn = block.attn
        scores = attn.last_attn_scores
        mask = attn.last_attn_mask
        
        if scores is not None:
            # Compute statistics per head
            B, H, T, T2 = scores.shape
            
            print(f"\nLayer {i}:")
            print(f"  Shape: {scores.shape}")
            
            for h in range(H):
                head_scores = scores[:, h, :, :]
                valid_scores = head_scores[head_scores != float('-inf')]
                
                # Compute N_eff for this head
                s_abs = torch.abs(valid_scores)
                s_sum = s_abs.sum()
                if s_sum > 1e-10:
                    omega = s_abs / s_sum
                    neff = (1.0 / (omega ** 2).sum()).item()
                else:
                    neff = 0
                
                if mask is not None:
                    head_sparsity = 1.0 - mask[:, h, :, :].float().mean().item()
                else:
                    head_sparsity = 0
                
                print(f"  Head {h}: N_eff={neff:8.1f} | Sparsity={head_sparsity*100:5.1f}% | "
                      f"Mean={valid_scores.mean().item():7.3f} | Std={valid_scores.std().item():7.3f}")

# Run analysis
analyze_per_layer_emp(model, val_loader, beta=1.0)

In [ ]:
def plot_neff_distribution(model: nn.Module, loader: DataLoader):
    """Plot N_eff distribution across layers and heads."""
    model.eval()
    model.set_emp(None)  # No pruning, just analyze
    
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    
    with torch.no_grad():
        _ = model(x)
    
    n_layers = len(model.transformer.h)
    n_heads = model.config.n_head
    
    neff_matrix = torch.zeros(n_layers, n_heads)
    
    for i, block in enumerate(model.transformer.h):
        scores = block.attn.last_attn_scores
        if scores is None:
            continue
            
        for h in range(n_heads):
            head_scores = scores[:, h, :, :]
            valid_scores = head_scores[head_scores != float('-inf')]
            
            s_abs = torch.abs(valid_scores)
            s_sum = s_abs.sum()
            if s_sum > 1e-10:
                omega = s_abs / s_sum
                neff = (1.0 / (omega ** 2).sum()).item()
            else:
                neff = 0
            
            neff_matrix[i, h] = neff
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(neff_matrix.cpu().numpy(), cmap='viridis', aspect='auto')
    
    ax.set_xlabel('Head')
    ax.set_ylabel('Layer')
    ax.set_title('N_eff Distribution Across Layers and Heads')
    ax.set_xticks(range(n_heads))
    ax.set_yticks(range(n_layers))
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('N_eff')
    
    plt.tight_layout()
    plt.savefig('./checkpoints/neff_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return neff_matrix

neff_matrix = plot_neff_distribution(model, val_loader)

## 8. Extended Ablation Study

In [ ]:
# Fine-grained beta sweep around the optimal region
fine_beta_list = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

print("\n" + "="*100)
print("FINE-GRAINED BETA ABLATION (Per-Head Mode)")
print("="*100)
print(f"{'Beta':>8} | {'Val Loss':>10} | {'Val PPL':>10} | {'Test Loss':>10} | {'Test PPL':>10} | "
      f"{'Sparsity':>10} | {'ΔPPL (Val)':>12} | {'ΔPPL (Test)':>12}")
print("-" * 110)

# Get baseline
model.set_emp(None)
baseline_val_loss, baseline_val_ppl = evaluate(model, val_loader)
baseline_test_loss, baseline_test_ppl = evaluate(model, test_loader)

fine_results = []
for beta in fine_beta_list:
    model.set_emp(beta, 'per_head')
    
    val_loss, val_ppl = evaluate(model, val_loader)
    test_loss, test_ppl = evaluate(model, test_loader)
    
    analysis = analyze_attention_emp(model, val_loader, beta, 'per_head')
    sparsity = analysis['avg_sparsity']
    
    delta_val = val_ppl - baseline_val_ppl
    delta_test = test_ppl - baseline_test_ppl
    
    fine_results.append({
        'beta': beta,
        'val_ppl': val_ppl,
        'test_ppl': test_ppl,
        'sparsity': sparsity,
        'delta_val_ppl': delta_val,
        'delta_test_ppl': delta_test,
    })
    
    print(f"{beta:8.2f} | {val_loss:10.4f} | {val_ppl:10.2f} | {test_loss:10.4f} | {test_ppl:10.2f} | "
          f"{sparsity*100:9.2f}% | {delta_val:+12.2f} | {delta_test:+12.2f}")

model.set_emp(None)

In [ ]:
# Save all results
all_results = {
    'config': config.__dict__,
    'baseline_val_ppl': baseline_val_ppl,
    'baseline_test_ppl': baseline_test_ppl,
    'main_experiments': results,
    'fine_grained_ablation': fine_results,
    'neff_matrix': neff_matrix.cpu().numpy().tolist(),
}

import json
with open('./checkpoints/emp_attention_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("\nResults saved to ./checkpoints/emp_attention_results.json")

## 9. Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

print(f"\nModel Configuration:")
print(f"  - Layers: {config.n_layer}")
print(f"  - Heads: {config.n_head}")
print(f"  - Embedding dim: {config.n_embd}")
print(f"  - Context length: {config.block_size}")
print(f"  - Vocab size: {config.vocab_size}")

print(f"\nBaseline Performance:")
print(f"  - Val PPL: {baseline_val_ppl:.2f}")
print(f"  - Test PPL: {baseline_test_ppl:.2f}")

# Find optimal beta for each mode
df_results = pd.DataFrame(results)
for mode in modes:
    mode_df = df_results[df_results['mode'] == mode]
    if len(mode_df) > 0:
        # Find beta with minimal PPL increase while having some sparsity
        mode_df_sparse = mode_df[mode_df['sparsity'] > 0.01]
        if len(mode_df_sparse) > 0:
            best_row = mode_df_sparse.loc[mode_df_sparse['delta_test_ppl'].idxmin()]
            print(f"\nBest {mode.upper()} setting:")
            print(f"  - Beta: {best_row['beta']:.2f}")
            print(f"  - Sparsity: {best_row['sparsity']*100:.1f}%")
            print(f"  - Test PPL: {best_row['test_ppl']:.2f} (Δ={best_row['delta_test_ppl']:+.2f})")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)
print("""
1. EMP can be effectively applied to transformer attention scores (pre-softmax)
2. Per-head EMP allows independent pruning of each attention head
3. Global EMP applies uniform pruning across all heads
4. Beta ≈ 1.0 typically provides a good balance between sparsity and performance
5. Lower beta values increase sparsity but may degrade performance
6. Higher beta values (>1.5) retain most attention patterns with minimal sparsity
""")

In [ ]:
# Final model state
model.set_emp(None)
print("\nModel reset to baseline (no EMP pruning)")
print(f"\nAll outputs saved to ./checkpoints/")
print("  - GPT_WikiText2_best.pth (model checkpoint)")
print("  - emp_attention_results.json (experiment results)")
print("  - emp_attention_results.png (visualization)")
print("  - neff_distribution.png (N_eff heatmap)")